# SST Super-Resolution Multi-Model Evaluation & Climate Change Intercomparison
This notebook performs a comprehensive evaluation and intercomparison of **all 7 super-resolution models** (Flow Matching and GAN variants) trained for Sea-Surface Temperature (SST) downscaling over the Australasian domain:

1. **Part 1: Multi-Model Test-Set Evaluation (OFAM Holdout 2011–2014)** — Seasonal climatologies (`groupby('time.season')`), annual extremes (TXx), Marine Heatwave (MHW) frequency, and spatial error metrics against native OFAM ground truth.
2. **Part 2: Historical Climatology Validation & Added Value (ACCESS-CM2 1980–1989 vs OFAM)** — Annual & seasonal biases, spatial RMSE, and added value (error reduction) over raw coarse-resolution GCM input.
3. **Part 3: Climate Change Signals & MHW Projections (SSP585 2080–2089 vs Historical 1980–1989)** — Multi-model warming patterns ($\Delta T$), seasonal breakdowns, warming underestimation/amplification analysis, and Marine Heatwave exceedance frequency projections.

> **Data Processing Note:** All climatologies, reductions, and metrics are computed natively with `xarray` operations (`.groupby('time.season')`, `.mean()`, `.max()`, `.quantile()`, `xr.concat`). Coarse-resolution fields are reduced natively on their coarse grid before spatial interpolation to preserve physical integrity.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import warnings
warnings.filterwarnings('ignore')

# Professional styling settings
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.facecolor'] = '#f8f9fa'
plt.rcParams['axes.facecolor'] = '#edf0f2'

## 0. Configuration & Multi-Model Registry
We register all available models and their corresponding evaluation / downscaled ACCESS-CM2 runs.

In [ ]:
BASE_DIR = '/esi/project/niwa03712/rampaln/PUBLICATIONS/2026/SSTDownscaling'
RUNS_DIR = os.path.join(BASE_DIR, 'runs')

# Model Registry: Directory Name -> Human Readable Display Name
MODEL_REGISTRY = {
    'flow_sr': 'Flow-SR (AB3-PC 75st)',
    'gan_sr_v2': 'GAN-SR v2',
    'gan_sr_v2_hist_rcp85_continue_220k': 'GAN-SR v2 (Hist/RCP85 220k)',
    'gan_sr_v2b_image_only_critic': 'GAN-SR v2b (Image Critic)',
    'gan_sr_v2b_hist_rcp85_continue_220k': 'GAN-SR v2b (Hist/RCP85 220k)',
    'gan_sr_v3_hard_consistency': 'GAN-SR v3 (Hard Cons.)',
    'gan_sr_v3_hist_rcp85_continue_220k': 'GAN-SR v3 (Hard Cons. 220k)',
}

# Ground truth OFAM dataset
OFAM_FILE = os.path.join(BASE_DIR, 'sst_10km_OFAM_historical_Australia.nc')

SEASONS = ['DJF', 'MAM', 'JJA', 'SON']
MODEL_KEYS = list(MODEL_REGISTRY.keys())

print(f'Registered {len(MODEL_KEYS)} models for multi-model intercomparison:')
for k, v in MODEL_REGISTRY.items():
    print(f'  • {k:<36} -> {v}')

## Helper Functions for Metrics & Regridding
Following spatial evaluation standards, coarse-resolution fields are evaluated on their native grid first before bilinear interpolation to the high-resolution target grid.

In [ ]:
def spatial_rmse(pred, truth):
    """Compute spatial Root Mean Square Error (scalar), ignoring NaNs."""
    diff = pred - truth
    return float(np.sqrt(np.nanmean(diff.values ** 2)))

def spatial_bias(pred, truth):
    """Compute spatial mean bias (scalar), ignoring NaNs."""
    return float(np.nanmean((pred - truth).values))

def spatial_mae(pred, truth):
    """Compute spatial Mean Absolute Error (scalar), ignoring NaNs."""
    return float(np.nanmean(np.abs((pred - truth).values)))

def spatial_corr(pred, truth):
    """Compute spatial pattern Pearson correlation coefficient, ignoring NaNs."""
    p = pred.values.ravel()
    t = truth.values.ravel()
    mask = np.isfinite(p) & np.isfinite(t)
    if mask.sum() < 10:
        return np.nan
    return float(np.corrcoef(p[mask], t[mask])[0, 1])

def interp_coarse_to_fine(da_coarse, fine_lat, fine_lon):
    """Bilinearly interpolate coarse DataArray (lat_lr, lon_lr) to fine grid (lat, lon)."""
    lat_dim = 'lat_lr' if 'lat_lr' in da_coarse.dims else ('lat' if 'lat' in da_coarse.dims else da_coarse.dims[-2])
    lon_dim = 'lon_lr' if 'lon_lr' in da_coarse.dims else ('lon' if 'lon' in da_coarse.dims else da_coarse.dims[-1])
    return da_coarse.interp({lat_dim: fine_lat, lon_dim: fine_lon}, method='linear').rename({lat_dim: 'lat', lon_dim: 'lon'})

def prepare_ofam(ds):
    """Standardize OFAM ground truth dataset dimensions and coordinates."""
    da = ds['temp'] if 'temp' in ds.data_vars else ds[list(ds.data_vars)[0]]
    if 'st_ocean' in da.dims:
        da = da.isel(st_ocean=0, drop=True)
    rename_map = {}
    for d in da.dims:
        if d == 'Time': rename_map[d] = 'time'
        elif d == 'yt_ocean': rename_map[d] = 'lat'
        elif d == 'xt_ocean': rename_map[d] = 'lon'
    if rename_map:
        da = da.rename(rename_map)
    return da

---
# Part 1: Multi-Model Test-Set Evaluation (OFAM Hold-out 2011–2014)
Here we evaluate all models on the 4-year holdout test set (2011–2014, 1,461 daily time steps):
- **Multi-Model Climatology**: Annual and seasonal (`groupby('time.season')`) mean SST fields.
- **Extreme SST (TXx)**: Annual maximum SST (`groupby('time.year').max().mean('year')`).
- **Marine Heatwave (MHW) Frequency**: Daily exceedance fraction above the 90th percentile threshold.
- **Multi-Model Intercomparison Table**: Spatial RMSE, Bias, MAE, Pattern Correlation, and Seasonal RMSE across all models.

In [ ]:
# Load holdout test-set evaluation files across all registered models
# Test set holdout years: strictly 2011-2014 (1,461 daily time steps)
TEST_YEARS = [2011, 2012, 2013, 2014]
test_gen_list = []
valid_test_models = []
ds_ref = None

for model in MODEL_KEYS:
    test_path = os.path.join(RUNS_DIR, model, 'evaluation', 'full_test_samples.nc')
    if not os.path.exists(test_path):
        test_path = os.path.join(RUNS_DIR, model, 'evaluation', 'full_test_samples_ab3_pc_75step.nc')
    
    if os.path.exists(test_path):
        ds_m = xr.open_dataset(test_path)
        da_m = ds_m['sst_generated'].sel(time=ds_m.time.dt.year.isin(TEST_YEARS))
        test_gen_list.append(da_m)
        valid_test_models.append(model)
        if ds_ref is None:
            ds_ref = ds_m
        print(f'Loaded test samples for: {model} (time={da_m.sizes["time"]})')
    else:
        print(f'Warning: Test samples file not found for: {model}')

# Concatenate all model predictions along a new "model_name" dimension
da_test_models = xr.concat(test_gen_list, dim='model_name')
da_test_models['model_name'] = valid_test_models

da_test_tgt = ds_ref['sst_target'].sel(time=ds_ref.time.dt.year.isin(TEST_YEARS))
da_test_coarse_lr = ds_ref['sst_coarse'].sel(time=ds_ref.time.dt.year.isin(TEST_YEARS))
fine_lat = da_test_tgt.lat.values
fine_lon = da_test_tgt.lon.values

print(f'\nCombined Test-Set DataArray shape: {da_test_models.shape} (models, time, lat, lon)')

In [ ]:
# 1. Annual and Seasonal Climatologies using xarray.groupby
clim_test_tgt = da_test_tgt.mean(dim='time').compute()
clim_test_tgt_season = da_test_tgt.groupby('time.season').mean(dim='time').compute()

# Coarse resolution: compute natively, then interpolate to fine grid
clim_test_coarse_lr = da_test_coarse_lr.mean(dim='time').compute()
clim_test_coarse = interp_coarse_to_fine(clim_test_coarse_lr, fine_lat, fine_lon)

clim_test_coarse_lr_season = da_test_coarse_lr.groupby('time.season').mean(dim='time').compute()
clim_test_coarse_season = interp_coarse_to_fine(clim_test_coarse_lr_season, fine_lat, fine_lon)

# All AI Models:
clim_test_models = da_test_models.mean(dim='time').compute()
clim_test_models_season = da_test_models.groupby('time.season').mean(dim='time').compute()

# 2. Extreme Sea Surface Temperatures (TXx)
txx_test_tgt = da_test_tgt.groupby('time.year').max(dim='time').mean(dim='year').compute()
txx_test_coarse_lr = da_test_coarse_lr.groupby('time.year').max(dim='time').mean(dim='year').compute()
txx_test_coarse = interp_coarse_to_fine(txx_test_coarse_lr, fine_lat, fine_lon)
txx_test_models = da_test_models.groupby('time.year').max(dim='time').mean(dim='year').compute()

# 3. Marine Heatwave (MHW) Exceedance Frequency (>90th percentile threshold of Ground Truth)
thresh_test_tgt = da_test_tgt.quantile(0.90, dim='time').compute()
thresh_test_coarse_lr = da_test_coarse_lr.quantile(0.90, dim='time').compute()

mhw_test_tgt = (da_test_tgt > thresh_test_tgt).mean(dim='time').compute()
mhw_test_coarse_lr = (da_test_coarse_lr > thresh_test_coarse_lr).mean(dim='time').compute()
mhw_test_coarse = interp_coarse_to_fine(mhw_test_coarse_lr, fine_lat, fine_lon)
mhw_test_models = (da_test_models > thresh_test_tgt).mean(dim='time').compute()

In [ ]:
# Plot Multi-Model Climatology Bias Maps against OFAM Target
n_models = len(valid_test_models)
fig, axes = plt.subplots(2, 4, figsize=(22, 11), sharex=True, sharey=True)
axes_flat = axes.flatten()

# Panel 0: Coarse Input Bias
bias_coarse = clim_test_coarse - clim_test_tgt
rmse_c = spatial_rmse(clim_test_coarse, clim_test_tgt)
bias_c_val = spatial_bias(clim_test_coarse, clim_test_tgt)

norm_bias = TwoSlopeNorm(vmin=-1.5, vcenter=0, vmax=1.5)
im0 = bias_coarse.plot(ax=axes_flat[0], cmap='RdBu_r', norm=norm_bias, add_colorbar=False)
axes_flat[0].set_title(f'Coarse Input (Bilinear)\nRMSE = {rmse_c:.4f}°C | Bias = {bias_c_val:.4f}°C', weight='bold')

# Panels 1..N: AI Models Bias
for i, model in enumerate(valid_test_models):
    ax = axes_flat[i + 1]
    bias_m = clim_test_models.sel(model_name=model) - clim_test_tgt
    rmse_m = spatial_rmse(bias_m + clim_test_tgt, clim_test_tgt)
    bias_m_val = spatial_bias(bias_m + clim_test_tgt, clim_test_tgt)
    
    bias_m.plot(ax=ax, cmap='RdBu_r', norm=norm_bias, add_colorbar=False)
    ax.set_title(f'{MODEL_REGISTRY.get(model, model)}\nRMSE = {rmse_m:.4f}°C | Bias = {bias_m_val:.4f}°C', weight='bold')

for ax in axes_flat:
    ax.set_xlabel('Longitude (°E)')
    ax.set_ylabel('Latitude (°N)')

# Unified Colorbar
cbar_ax = fig.add_axes([0.25, 0.04, 0.50, 0.02])
cbar = fig.colorbar(im0, cax=cbar_ax, orientation='horizontal')
cbar.set_label('SST Climatology Bias (°C) relative to OFAM Ground Truth', fontsize=11)

plt.suptitle('Test Set (OFAM Hold-out 2011–2014): Multi-Model Climatology Bias Maps', fontsize=16, weight='bold', y=0.98)
plt.subplots_adjust(bottom=0.10, top=0.92, hspace=0.25, wspace=0.10)
plt.show()

In [ ]:
# Multi-Model Seasonal Bias Intercomparison across DJF, MAM, JJA, SON
fig, axes = plt.subplots(4, len(valid_test_models) + 1, figsize=(3 * (len(valid_test_models) + 1), 16), sharex=True, sharey=True)

norm_s = TwoSlopeNorm(vmin=-2.0, vcenter=0, vmax=2.0)

for r, s in enumerate(SEASONS):
    # Coarse column
    b_c_s = clim_test_coarse_season.sel(season=s) - clim_test_tgt_season.sel(season=s)
    b_c_s.plot(ax=axes[r, 0], cmap='RdBu_r', norm=norm_s, add_colorbar=False)
    if r == 0:
        axes[r, 0].set_title('Coarse Input', weight='bold')
    axes[r, 0].set_ylabel(f'{s}\nLatitude (°N)', weight='bold')
    
    # AI Models columns
    for c, model in enumerate(valid_test_models):
        ax = axes[r, c + 1]
        b_m_s = clim_test_models_season.sel(model_name=model, season=s) - clim_test_tgt_season.sel(season=s)
        im = b_m_s.plot(ax=ax, cmap='RdBu_r', norm=norm_s, add_colorbar=False)
        if r == 0:
            ax.set_title(MODEL_REGISTRY.get(model, model), fontsize=9.5, weight='bold')
        if r == 3:
            ax.set_xlabel('Longitude (°E)')
        else:
            ax.set_xlabel('')

cbar_ax = fig.add_axes([0.25, 0.03, 0.50, 0.015])
cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal')
cbar.set_label('Seasonal SST Bias (°C) vs OFAM Target', fontsize=11)

plt.suptitle('Test Set: Multi-Model Seasonal Climatology Bias Breakdown', fontsize=16, weight='bold', y=0.99)
plt.subplots_adjust(bottom=0.07, top=0.95, hspace=0.18, wspace=0.08)
plt.show()

In [ ]:
# Tabulate Comprehensive Part 1 Test-Set Metrics for All Models
metrics_dict = {}

# 1. Coarse Input Baseline
metrics_dict['Coarse Input'] = {
    'Annual RMSE (°C)': spatial_rmse(clim_test_coarse, clim_test_tgt),
    'Annual Bias (°C)': spatial_bias(clim_test_coarse, clim_test_tgt),
    'Annual MAE (°C)': spatial_mae(clim_test_coarse, clim_test_tgt),
    'Pattern Correlation': spatial_corr(clim_test_coarse, clim_test_tgt),
    'TXx Error (°C)': spatial_mae(txx_test_coarse, txx_test_tgt),
    'MHW Freq RMSE': spatial_rmse(mhw_test_coarse, mhw_test_tgt),
    'DJF RMSE (°C)': spatial_rmse(clim_test_coarse_season.sel(season='DJF'), clim_test_tgt_season.sel(season='DJF')),
    'MAM RMSE (°C)': spatial_rmse(clim_test_coarse_season.sel(season='MAM'), clim_test_tgt_season.sel(season='MAM')),
    'JJA RMSE (°C)': spatial_rmse(clim_test_coarse_season.sel(season='JJA'), clim_test_tgt_season.sel(season='JJA')),
    'SON RMSE (°C)': spatial_rmse(clim_test_coarse_season.sel(season='SON'), clim_test_tgt_season.sel(season='SON')),
}

# 2. All AI Models
for model in valid_test_models:
    m_name = MODEL_REGISTRY.get(model, model)
    clim_m = clim_test_models.sel(model_name=model)
    txx_m = txx_test_models.sel(model_name=model)
    mhw_m = mhw_test_models.sel(model_name=model)
    clim_m_s = clim_test_models_season.sel(model_name=model)
    
    metrics_dict[m_name] = {
        'Annual RMSE (°C)': spatial_rmse(clim_m, clim_test_tgt),
        'Annual Bias (°C)': spatial_bias(clim_m, clim_test_tgt),
        'Annual MAE (°C)': spatial_mae(clim_m, clim_test_tgt),
        'Pattern Correlation': spatial_corr(clim_m, clim_test_tgt),
        'TXx Error (°C)': spatial_mae(txx_m, txx_test_tgt),
        'MHW Freq RMSE': spatial_rmse(mhw_m, mhw_test_tgt),
        'DJF RMSE (°C)': spatial_rmse(clim_m_s.sel(season='DJF'), clim_test_tgt_season.sel(season='DJF')),
        'MAM RMSE (°C)': spatial_rmse(clim_m_s.sel(season='MAM'), clim_test_tgt_season.sel(season='MAM')),
        'JJA RMSE (°C)': spatial_rmse(clim_m_s.sel(season='JJA'), clim_test_tgt_season.sel(season='JJA')),
        'SON RMSE (°C)': spatial_rmse(clim_m_s.sel(season='SON'), clim_test_tgt_season.sel(season='SON')),
    }

df_part1 = pd.DataFrame(metrics_dict).T.round(4)
print('=== Part 1: Multi-Model Test-Set Performance Summary ===')
display(df_part1)

---
# Part 2: Historical Climatology Validation & Added Value (ACCESS-CM2 1980–1989 vs OFAM)
Here we validate the downscaling of ACCESS-CM2 over the 10 common historical years (1980–1989) against OFAM ground truth:
- **Multi-Model Climatology**: 10-year mean SST field from OFAM ground truth, raw coarse ACCESS-CM2, and all downscaled AI models.
- **Added Value of Downscaling**: Spatial bias patterns, error reduction over raw GCM input, and seasonal consistency.

In [ ]:
# Load OFAM historical ground truth and subset 10 common years
ds_ofam = xr.open_dataset(OFAM_FILE)
da_ofam = prepare_ofam(ds_ofam)

hist_models_list = []
valid_hist_models = []
da_hist_coarse_lr = None

for model in MODEL_KEYS:
    pattern = os.path.join(RUNS_DIR, model, 'access_cm2_converted', 'historical_*.nc')
    matching_files = glob.glob(pattern)
    if matching_files:
        ds_h = xr.open_dataset(matching_files[0])
        hist_models_list.append(ds_h['sst_downscaled'])
        valid_hist_models.append(model)
        if da_hist_coarse_lr is None:
            da_hist_coarse_lr = ds_h['sst_coarse']
        print(f'Loaded historical ACCESS-CM2 downscaling: {model} ({matching_files[0].split("/")[-1]})')
    else:
        print(f'Warning: Historical file missing for: {model}')

da_hist_models = xr.concat(hist_models_list, dim='model_name')
da_hist_models['model_name'] = valid_hist_models

# Find common historical years (1980-1989)
common_years = np.intersect1d(np.unique(da_hist_models.time.dt.year.values), np.unique(da_ofam.time.dt.year.values))
print(f'\nIdentified {len(common_years)} Common Historical Years: {common_years}')

# Subset to common years
da_hist_models_common = da_hist_models.sel(time=da_hist_models.time.dt.year.isin(common_years))
da_hist_coarse_common = da_hist_coarse_lr.sel(time=da_hist_coarse_lr.time.dt.year.isin(common_years))
da_ofam_common = da_ofam.sel(time=da_ofam.time.dt.year.isin(common_years))

In [ ]:
# Compute 10-year mean historical climatologies (Annual & Seasonal)
clim_hist_ofam = da_ofam_common.mean(dim='time').compute()
clim_hist_ofam_season = da_ofam_common.groupby('time.season').mean(dim='time').compute()

# Coarse GCM climatology computed natively then regridded
clim_hist_coarse_lr = da_hist_coarse_common.mean(dim='time').compute()
clim_hist_coarse = interp_coarse_to_fine(clim_hist_coarse_lr, da_hist_models.lat.values, da_hist_models.lon.values)

clim_hist_coarse_lr_season = da_hist_coarse_common.groupby('time.season').mean(dim='time').compute()
clim_hist_coarse_season = interp_coarse_to_fine(clim_hist_coarse_lr_season, da_hist_models.lat.values, da_hist_models.lon.values)

# AI Downscaled models climatologies
clim_hist_models = da_hist_models_common.mean(dim='time').compute()
clim_hist_models_season = da_hist_models_common.groupby('time.season').mean(dim='time').compute()

In [ ]:
# Plot Multi-Model Historical Biases relative to OFAM Ground Truth
fig, axes = plt.subplots(2, 4, figsize=(22, 11), sharex=True, sharey=True)
axes_flat = axes.flatten()

# Panel 0: Raw Coarse ACCESS-CM2 Bias
bias_hist_coarse = clim_hist_coarse - clim_hist_ofam
rmse_h_c = spatial_rmse(clim_hist_coarse, clim_hist_ofam)
bias_h_c_val = spatial_bias(clim_hist_coarse, clim_hist_ofam)

norm_hist_bias = TwoSlopeNorm(vmin=-2.5, vcenter=0, vmax=2.5)
im0 = bias_hist_coarse.plot(ax=axes_flat[0], cmap='RdBu_r', norm=norm_hist_bias, add_colorbar=False)
axes_flat[0].set_title(f'Raw Coarse ACCESS-CM2\nRMSE = {rmse_h_c:.4f}°C | Bias = {bias_h_c_val:.4f}°C', weight='bold')

# Panels 1..N: Downscaled AI Models
for i, model in enumerate(valid_hist_models):
    ax = axes_flat[i + 1]
    bias_h_m = clim_hist_models.sel(model_name=model) - clim_hist_ofam
    rmse_h_m = spatial_rmse(bias_h_m + clim_hist_ofam, clim_hist_ofam)
    bias_h_m_val = spatial_bias(bias_h_m + clim_hist_ofam, clim_hist_ofam)
    
    bias_h_m.plot(ax=ax, cmap='RdBu_r', norm=norm_hist_bias, add_colorbar=False)
    ax.set_title(f'{MODEL_REGISTRY.get(model, model)}\nRMSE = {rmse_h_m:.4f}°C | Bias = {bias_h_m_val:.4f}°C', weight='bold')

for ax in axes_flat:
    ax.set_xlabel('Longitude (°E)')
    ax.set_ylabel('Latitude (°N)')

cbar_ax = fig.add_axes([0.25, 0.04, 0.50, 0.02])
cbar = fig.colorbar(im0, cax=cbar_ax, orientation='horizontal')
cbar.set_label('Historical SST Bias (°C) relative to OFAM Ground Truth (1980–1989)', fontsize=11)

plt.suptitle('Historical Climatology Validation (ACCESS-CM2 1980–1989 vs OFAM)', fontsize=16, weight='bold', y=0.98)
plt.subplots_adjust(bottom=0.10, top=0.92, hspace=0.25, wspace=0.10)
plt.show()

In [ ]:
# Tabulate Part 2 Historical Metrics & Added Value of Downscaling
hist_metrics = {}

# Baseline
rmse_base = spatial_rmse(clim_hist_coarse, clim_hist_ofam)
hist_metrics['Raw Coarse ACCESS-CM2'] = {
    'Annual RMSE (°C)': rmse_base,
    'Annual Bias (°C)': spatial_bias(clim_hist_coarse, clim_hist_ofam),
    'Pattern Corr': spatial_corr(clim_hist_coarse, clim_hist_ofam),
    'Error Reduction (%)': 0.0,
    'DJF RMSE (°C)': spatial_rmse(clim_hist_coarse_season.sel(season='DJF'), clim_hist_ofam_season.sel(season='DJF')),
    'MAM RMSE (°C)': spatial_rmse(clim_hist_coarse_season.sel(season='MAM'), clim_hist_ofam_season.sel(season='MAM')),
    'JJA RMSE (°C)': spatial_rmse(clim_hist_coarse_season.sel(season='JJA'), clim_hist_ofam_season.sel(season='JJA')),
    'SON RMSE (°C)': spatial_rmse(clim_hist_coarse_season.sel(season='SON'), clim_hist_ofam_season.sel(season='SON')),
}

for model in valid_hist_models:
    m_name = MODEL_REGISTRY.get(model, model)
    clim_m = clim_hist_models.sel(model_name=model)
    clim_m_s = clim_hist_models_season.sel(model_name=model)
    rmse_m = spatial_rmse(clim_m, clim_hist_ofam)
    err_reduc = ((rmse_base - rmse_m) / rmse_base) * 100.0
    
    hist_metrics[m_name] = {
        'Annual RMSE (°C)': rmse_m,
        'Annual Bias (°C)': spatial_bias(clim_m, clim_hist_ofam),
        'Pattern Corr': spatial_corr(clim_m, clim_hist_ofam),
        'Error Reduction (%)': err_reduc,
        'DJF RMSE (°C)': spatial_rmse(clim_m_s.sel(season='DJF'), clim_hist_ofam_season.sel(season='DJF')),
        'MAM RMSE (°C)': spatial_rmse(clim_m_s.sel(season='MAM'), clim_hist_ofam_season.sel(season='MAM')),
        'JJA RMSE (°C)': spatial_rmse(clim_m_s.sel(season='JJA'), clim_hist_ofam_season.sel(season='JJA')),
        'SON RMSE (°C)': spatial_rmse(clim_m_s.sel(season='SON'), clim_hist_ofam_season.sel(season='SON')),
    }

df_part2 = pd.DataFrame(hist_metrics).T.round(4)
print('=== Part 2: Historical Added Value Performance Summary ===')
display(df_part2)

---
# Part 3: Climate Change Signals & MHW Projections (SSP585 2080–2089 vs Historical 1980–1989)
Here we quantify the climate change warming response (SSP585 2080–2089 minus Historical 1980–1989) across all models:
- **Multi-Model Warming Signals ($\Delta T$)**: Annual and seasonal spatial warming distributions.
- **Warming Discrepancy ($\Delta T_{\text{downscaled}} - \Delta T_{\text{coarse}}$)**: Identifying regions of warming amplification or underestimation.
- **Marine Heatwave (MHW) Future Projections**: Exceedance frequency relative to the historical 90th percentile baseline and change in extreme days ($\Delta \text{MHW}$).
- **Multi-Model Synthesis Table**: Summary of regional climate change statistics.

In [ ]:
# Load future ACCESS-CM2 downscaled runs across all models
fut_models_list = []
valid_fut_models = []
da_fut_coarse_lr = None

for model in MODEL_KEYS:
    pattern = os.path.join(RUNS_DIR, model, 'access_cm2_converted', 'future_*.nc')
    matching_files = glob.glob(pattern)
    if matching_files:
        ds_f = xr.open_dataset(matching_files[0])
        fut_models_list.append(ds_f['sst_downscaled'])
        valid_fut_models.append(model)
        if da_fut_coarse_lr is None:
            da_fut_coarse_lr = ds_f['sst_coarse']
        print(f'Loaded future ACCESS-CM2 downscaling: {model} ({matching_files[0].split("/")[-1]})')
    else:
        print(f'Warning: Future file missing for: {model}')

da_fut_models = xr.concat(fut_models_list, dim='model_name')
da_fut_models['model_name'] = valid_fut_models

print(f'\nFuture Multi-Model DataArray shape: {da_fut_models.shape}')

In [ ]:
# 1. Compute Mean Climatologies for Future and Historical Periods
clim_fut_models = da_fut_models.mean(dim='time').compute()
clim_hist_models_all = da_hist_models.mean(dim='time').compute()

clim_fut_coarse_lr = da_fut_coarse_lr.mean(dim='time').compute()
clim_hist_coarse_lr_all = da_hist_coarse_lr.mean(dim='time').compute()

clim_fut_coarse = interp_coarse_to_fine(clim_fut_coarse_lr, da_fut_models.lat.values, da_fut_models.lon.values)
clim_hist_coarse_all = interp_coarse_to_fine(clim_hist_coarse_lr_all, da_hist_models.lat.values, da_hist_models.lon.values)

# 2. Climate Change Warming Signals (ΔT)
delta_coarse = (clim_fut_coarse - clim_hist_coarse_all).compute()
delta_models = (clim_fut_models - clim_hist_models_all).compute()
delta_diff_models = (delta_models - delta_coarse).compute()

# 3. Seasonal Warming Signals
clim_fut_models_season = da_fut_models.groupby('time.season').mean(dim='time').compute()
clim_hist_models_season_all = da_hist_models.groupby('time.season').mean(dim='time').compute()
delta_models_season = (clim_fut_models_season - clim_hist_models_season_all).compute()

clim_fut_coarse_lr_season = da_fut_coarse_lr.groupby('time.season').mean(dim='time').compute()
clim_hist_coarse_lr_season_all = da_hist_coarse_lr.groupby('time.season').mean(dim='time').compute()
delta_coarse_season = interp_coarse_to_fine(clim_fut_coarse_lr_season - clim_hist_coarse_lr_season_all, da_fut_models.lat.values, da_fut_models.lon.values).compute()

# 4. Future Marine Heatwave (MHW) Projections
# Threshold: Historical 90th percentile
thresh_hist_models = da_hist_models.quantile(0.90, dim='time').compute()
thresh_hist_coarse_lr = da_hist_coarse_lr.quantile(0.90, dim='time').compute()

mhw_hist_models = (da_hist_models > thresh_hist_models).mean(dim='time').compute()
mhw_fut_models = (da_fut_models > thresh_hist_models).mean(dim='time').compute()
mhw_delta_models = (mhw_fut_models - mhw_hist_models).compute()

mhw_hist_coarse_lr = (da_hist_coarse_lr > thresh_hist_coarse_lr).mean(dim='time').compute()
mhw_fut_coarse_lr = (da_fut_coarse_lr > thresh_hist_coarse_lr).mean(dim='time').compute()
mhw_delta_coarse_lr = mhw_fut_coarse_lr - mhw_hist_coarse_lr
mhw_delta_coarse = interp_coarse_to_fine(mhw_delta_coarse_lr, da_fut_models.lat.values, da_fut_models.lon.values).compute()

In [ ]:
# Plot Multi-Model Climate Change Warming Signal (ΔT)
fig, axes = plt.subplots(2, 4, figsize=(22, 11), sharex=True, sharey=True)
axes_flat = axes.flatten()

vmin_w = float(min(delta_coarse.min(), delta_models.min()))
vmax_w = float(max(delta_coarse.max(), delta_models.max()))

# Panel 0: Coarse ACCESS-CM2 ΔT
im0 = delta_coarse.plot(ax=axes_flat[0], cmap='RdYlBu_r', vmin=vmin_w, vmax=vmax_w, add_colorbar=False)
axes_flat[0].set_title(f'Raw Coarse ACCESS-CM2 ΔT\nMean = {float(np.nanmean(delta_coarse)):.3f}°C', weight='bold')

# Panels 1..N: AI Models ΔT
for i, model in enumerate(valid_fut_models):
    ax = axes_flat[i + 1]
    delta_m = delta_models.sel(model_name=model)
    diff_m = delta_diff_models.sel(model_name=model)
    
    delta_m.plot(ax=ax, cmap='RdYlBu_r', vmin=vmin_w, vmax=vmax_w, add_colorbar=False)
    ax.set_title(f'{MODEL_REGISTRY.get(model, model)} ΔT\nMean = {float(np.nanmean(delta_m)):.3f}°C (Diff = {float(np.nanmean(diff_m)):+.3f}°C)', weight='bold')

for ax in axes_flat:
    ax.set_xlabel('Longitude (°E)')
    ax.set_ylabel('Latitude (°N)')

cbar_ax = fig.add_axes([0.25, 0.04, 0.50, 0.02])
cbar = fig.colorbar(im0, cax=cbar_ax, orientation='horizontal')
cbar.set_label('Climate Change Warming Signal ΔT (°C) [2080–2089 minus 1980–1989]', fontsize=11)

plt.suptitle('Multi-Model Climate Change Projections: SSP585 (2080–2089) − Historical (1980–1989)', fontsize=16, weight='bold', y=0.98)
plt.subplots_adjust(bottom=0.10, top=0.92, hspace=0.25, wspace=0.10)
plt.show()

In [ ]:
# Plot Warming Discrepancy Maps (Downscaled AI minus Raw Coarse ΔT)
fig, axes = plt.subplots(2, 4, figsize=(22, 11), sharex=True, sharey=True)
axes_flat = axes.flatten()

# Hide panel 0 or use for domain context
axes_flat[0].axis('off')
axes_flat[0].text(0.5, 0.5, 'Warming Discrepancy\nΔT_downscaled − ΔT_coarse\n\nRed = Amplified Warming\nBlue = Dampened Warming', 
                  ha='center', va='center', fontsize=13, weight='bold', color='#102a43')

norm_diff = TwoSlopeNorm(vmin=-0.8, vcenter=0, vmax=0.8)

for i, model in enumerate(valid_fut_models):
    ax = axes_flat[i + 1]
    diff_m = delta_diff_models.sel(model_name=model)
    mean_diff = float(np.nanmean(diff_m))
    underest_pct = (mean_diff / float(np.nanmean(delta_coarse))) * 100.0
    
    im = diff_m.plot(ax=ax, cmap='RdBu_r', norm=norm_diff, add_colorbar=False)
    ax.set_title(f'{MODEL_REGISTRY.get(model, model)}\nMean Diff = {mean_diff:+.3f}°C ({underest_pct:+.1f}%)', weight='bold')
    ax.set_xlabel('Longitude (°E)')
    ax.set_ylabel('Latitude (°N)')

cbar_ax = fig.add_axes([0.25, 0.04, 0.50, 0.02])
cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal')
cbar.set_label('Warming Difference (Downscaled − Coarse ΔT, °C)', fontsize=11)

plt.suptitle('Multi-Model Warming Discrepancy Maps (Downscaled AI vs Raw Coarse ACCESS-CM2)', fontsize=16, weight='bold', y=0.98)
plt.subplots_adjust(bottom=0.10, top=0.92, hspace=0.25, wspace=0.10)
plt.show()

In [ ]:
# Plot Multi-Model Change in Marine Heatwave Exceedance Frequency (ΔMHW)
fig, axes = plt.subplots(2, 4, figsize=(22, 11), sharex=True, sharey=True)
axes_flat = axes.flatten()

norm_mhw = TwoSlopeNorm(vmin=-0.2, vcenter=0.5, vmax=1.0)

# Panel 0: Coarse ΔMHW
im0 = mhw_delta_coarse.plot(ax=axes_flat[0], cmap='YlOrRd', vmin=0, vmax=0.9, add_colorbar=False)
axes_flat[0].set_title(f'Raw Coarse ACCESS-CM2 ΔMHW\nMean Δ = +{float(np.nanmean(mhw_delta_coarse)):.3f}', weight='bold')

for i, model in enumerate(valid_fut_models):
    ax = axes_flat[i + 1]
    mhw_d_m = mhw_delta_models.sel(model_name=model)
    
    mhw_d_m.plot(ax=ax, cmap='YlOrRd', vmin=0, vmax=0.9, add_colorbar=False)
    ax.set_title(f'{MODEL_REGISTRY.get(model, model)}\nMean Δ = +{float(np.nanmean(mhw_d_m)):.3f}', weight='bold')
    ax.set_xlabel('Longitude (°E)')
    ax.set_ylabel('Latitude (°N)')

axes_flat[0].set_xlabel('Longitude (°E)')
axes_flat[0].set_ylabel('Latitude (°N)')

cbar_ax = fig.add_axes([0.25, 0.04, 0.50, 0.02])
cbar = fig.colorbar(im0, cax=cbar_ax, orientation='horizontal')
cbar.set_label('Change in MHW Exceedance Frequency (Δ Fraction of Days exceeding Historical 90th %ile)', fontsize=11)

plt.suptitle('Multi-Model Marine Heatwave (MHW) Projections: End-of-Century Frequency Shift', fontsize=16, weight='bold', y=0.98)
plt.subplots_adjust(bottom=0.10, top=0.92, hspace=0.25, wspace=0.10)
plt.show()

In [ ]:
# Multi-Model Climate Change Synthesis Table
cc_summary = {}

# 1. Coarse GCM
mean_c_w = float(np.nanmean(delta_coarse))
cc_summary['Raw Coarse ACCESS-CM2'] = {
    'Annual Mean ΔT (°C)': mean_c_w,
    'Max Pixel ΔT (°C)': float(np.nanmax(delta_coarse)),
    'DJF ΔT (°C)': float(np.nanmean(delta_coarse_season.sel(season='DJF'))),
    'MAM ΔT (°C)': float(np.nanmean(delta_coarse_season.sel(season='MAM'))),
    'JJA ΔT (°C)': float(np.nanmean(delta_coarse_season.sel(season='JJA'))),
    'SON ΔT (°C)': float(np.nanmean(delta_coarse_season.sel(season='SON'))),
    'Warming Diff (°C)': 0.0,
    'Underestimation (%)': 0.0,
    'Mean ΔMHW Fraction': float(np.nanmean(mhw_delta_coarse)),
}

# 2. All AI Models
for model in valid_fut_models:
    m_name = MODEL_REGISTRY.get(model, model)
    delta_m = delta_models.sel(model_name=model)
    delta_m_s = delta_models_season.sel(model_name=model)
    diff_m = delta_diff_models.sel(model_name=model)
    mhw_d_m = mhw_delta_models.sel(model_name=model)
    
    mean_m_w = float(np.nanmean(delta_m))
    mean_diff = float(np.nanmean(diff_m))
    underest = (mean_diff / mean_c_w) * 100.0
    
    cc_summary[m_name] = {
        'Annual Mean ΔT (°C)': mean_m_w,
        'Max Pixel ΔT (°C)': float(np.nanmax(delta_m)),
        'DJF ΔT (°C)': float(np.nanmean(delta_m_s.sel(season='DJF'))),
        'MAM ΔT (°C)': float(np.nanmean(delta_m_s.sel(season='MAM'))),
        'JJA ΔT (°C)': float(np.nanmean(delta_m_s.sel(season='JJA'))),
        'SON ΔT (°C)': float(np.nanmean(delta_m_s.sel(season='SON'))),
        'Warming Diff (°C)': mean_diff,
        'Underestimation (%)': underest,
        'Mean ΔMHW Fraction': float(np.nanmean(mhw_d_m)),
    }

df_part3 = pd.DataFrame(cc_summary).T.round(3)
print('=== Part 3: Climate Change Signals & MHW Projections Summary ===')
display(df_part3)


---

# Validated extension: temporal memory and 0.05° satellite transfer

This section was generated from independently validated final NetCDF products.
It intentionally does **not** concatenate the NOAA-transfer output with the
0.1° OFAM model registry: the target product, grid, ocean mask, and test years
are different. The autoregressive comparison is also time-aligned to the common
364 generated days in 2011 and has no truth resets.

## 2011 autoregressive comparison

| Model | RMSE (°C) | Bias (°C) | Evolution ratio | EAC r |
|---|---:|---:|---:|---:|
| Flow-SR (no memory) | 0.475 | -0.021 | 2.78 | 0.948 |
| Legacy Flow-AR | 0.539 | -0.003 | 1.26 | 0.884 |
| Residual-memory Flow-AR | 0.519 | -0.000 | 1.10 | 0.956 |
| Coarse-balanced legacy Flow-AR | 0.621 | 0.008 | 0.81 | 0.936 |
| GAN-v2 (historical) | 0.492 | -0.001 | 1.02 | 0.928 |
| GAN-v2b (historical) | 0.495 | -0.002 | 1.05 | 0.931 |
| GAN-v3 (historical) | 0.490 | -0.000 | 0.99 | 0.924 |
| GAN-v2 (historical + future) | 0.492 | -0.001 | 1.03 | 0.934 |
| GAN-v2b (historical + future) | 0.495 | -0.004 | 1.06 | 0.938 |
| GAN-v3 (historical + future) | 0.490 | -0.001 | 1.00 | 0.925 |

Interpretation: the useful autoregressive model must improve temporal coherence
without allowing yesterday's state to override today's coarse SST. The residual
memory experiment freezes the successful current-SST backbone, conditions only
on yesterday's within-block anomaly, caps its FiLM contribution, and projects
every generated day to the current coarse ocean-block means. Common latent noise
couples stochastic texture through time without changing daily marginals.

The GAN rows use the same 364 dates and truth, but they are direct conditional
samples rather than free-running rollouts. Their evolution ratio therefore
measures frame-to-frame variability (including stochastic texture), not memory
stability. This is why RMSE, evolution ratio and regional correlation must be
read together rather than collapsed into one ranking.

The accompanying coarse-balanced Flow-AR animation uses the validated,
truth-reset-free 2011 rollout and a fixed SST colour scale. The Australian domain and the
Perth/southwest, Ningaloo/northwest shelf, and East Australian Current insets
show whether memory follows both today's coarse boundary and OFAM truth.

## NOAA 0.05° transfer and climate deployment

The final 0.05° model has daily test RMSE **0.385 °C**, 1-pixel coastal RMSE **0.375 °C**, and >8-pixel interior RMSE **0.393 °C**. The downscaled ACCESS-CM2 2080s−1980s domain-mean signal is **2.379 °C**.

Coastline metrics are reported separately because the failed 2×2 projection
experiment mixed incompatible NOAA and OFAM masks. The replacement predicts the
1024×1024 NOAA field directly and evaluates only on the fixed NOAA ocean mask.


In [ ]:
from pathlib import Path
from IPython.display import Image, Video, display

extended_figure_dir = Path(BASE_DIR if 'BASE_DIR' in globals() else '.') / 'figures/extended_evaluation'
for filename in (
    'autoregressive_2011_skill_and_eac_timeseries.png',
    'autoregressive_2011_mean_bias_maps.png',
    'noaa_5km_test_climatology_and_bias.png',
    'noaa_5km_access_cm2_warming_signal.png',
):
    path = extended_figure_dir / filename
    if path.is_file():
        display(Image(filename=str(path)))

animation_dir = Path(BASE_DIR if 'BASE_DIR' in globals() else '.') / 'figures'
preview = animation_dir / 'flow_ar_legacy_coarse_balanced_sst_preview.gif'
video = animation_dir / 'flow_ar_legacy_coarse_balanced_sst_comparison_300frames.mp4'
if preview.is_file():
    display(Image(filename=str(preview)))
if video.is_file():
    display(Video(filename=str(video), embed=False, html_attributes='controls loop'))


In [ ]:
import json
from pathlib import Path

report_path = Path(BASE_DIR if 'BASE_DIR' in globals() else '.') / 'reports/extended_evaluation/extended_model_comparison.json'
extended_report = json.loads(report_path.read_text())
extended_report


{
  "status": "passed",
  "ar": {
    "Flow-SR (no memory)": {
      "days": 364,
      "rmse_c": 0.4746304718543245,
      "mae_c": 0.3196374161930789,
      "bias_c": -0.021227607127008198,
      "climatology_rmse_c": 0.14051650706271054,
      "generated_mean_abs_daily_change_c": 0.1887890844318203,
      "target_mean_abs_daily_change_c": 0.06781965352297098,
      "evolution_ratio": 2.783692847499968,
      "point_correlations": {
        "EAC": 0.9481517107210526,
        "Ningaloo": 0.9837961217522991
      }
    },
    "Legacy Flow-AR": {
      "days": 364,
      "rmse_c": 0.5387900644789654,
      "mae_c": 0.36176468656313304,
      "bias_c": -0.0032079820564387897,
      "climatology_rmse_c": 0.1894616327940454,
      "generated_mean_abs_daily_change_c": 0.08544521102863444,
      "target_mean_abs_daily_change_c": 0.06781965352297098,
      "evolution_ratio": 1.2598886397981015,
      "point_correlations": {
        "EAC": 0.884193623727734,
        "Ningaloo": 0.9834161300630


---

# Validated climate-change evaluation

Two questions are deliberately separated. In the **perfect OFAM framework**,
the generated fine grid can be compared with paired fine-resolution truth. In
the **imperfect ACCESS-CM2 deployment**, no fine-resolution truth exists, so
the test is whether the generated climate-change field preserves the supplied
32×32 ACCESS-CM2 signal after exact mask-aware re-coarsening.

| Framework | Model | Historical | Future | Reference ΔT | Predicted ΔT | Ratio | Signal RMSE | Spatial r |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| Perfect OFAM | Flow-SR, historical + future | 2011–2014 | 2098–2101 | 2.729 °C | 2.722 °C | 0.997 | 0.085 °C | 0.993 |
| Perfect OFAM | GAN-v2, historical + future | 2011–2014 | 2098–2101 | 2.729 °C | 2.733 °C | 1.001 | 0.133 °C | 0.983 |
| Perfect OFAM | GAN-v2b, historical + future | 2011–2014 | 2098–2101 | 2.729 °C | 2.736 °C | 1.003 | 0.128 °C | 0.985 |
| Perfect OFAM | GAN-v3, historical + future | 2011–2014 | 2098–2101 | 2.729 °C | 2.730 °C | 1.000 | 0.131 °C | 0.984 |
| Imperfect ACCESS-CM2 | Flow-SR, historical + future | 1980–1989 | 2080–2089 | 2.918 °C | 2.894 °C | 0.992 | 0.030 °C | 1.000 |
| Imperfect ACCESS-CM2 | Flow-SR, historical only | 1980–1989 | 2080–2089 | 2.918 °C | 2.307 °C | 0.791 | 0.658 °C | 0.905 |
| Imperfect ACCESS-CM2 | NOAA 0.05° Flow-SR transfer | 1980–1989 | 2080–2089 | 2.918 °C | 2.406 °C | 0.824 | 0.552 °C | 0.935 |
| Imperfect ACCESS-CM2 | GAN-v2, historical + future | 1980–1989 | 2080–2089 | 2.918 °C | 2.922 °C | 1.001 | 0.017 °C | 1.000 |
| Imperfect ACCESS-CM2 | GAN-v2, historical only | 1980–1989 | 2080–2089 | 2.918 °C | 2.931 °C | 1.004 | 0.041 °C | 0.998 |
| Imperfect ACCESS-CM2 | GAN-v2b, historical + future | 1980–1989 | 2080–2089 | 2.918 °C | 2.926 °C | 1.003 | 0.035 °C | 0.998 |
| Imperfect ACCESS-CM2 | GAN-v2b, historical only | 1980–1989 | 2080–2089 | 2.918 °C | 2.936 °C | 1.006 | 0.060 °C | 0.995 |
| Imperfect ACCESS-CM2 | GAN-v3, historical + future | 1980–1989 | 2080–2089 | 2.918 °C | 2.918 °C | 1.000 | 0.000 °C | 1.000 |
| Imperfect ACCESS-CM2 | GAN-v3, historical only | 1980–1989 | 2080–2089 | 2.918 °C | 2.918 °C | 1.000 | 0.000 °C | 1.000 |

The combined model's daily RMSE is **0.473 °C** on the
2011–2014 OFAM test and **0.521 °C** on the 2098–2101 test.
Its predicted OFAM mean warming differs from truth by only
**-0.008 °C**. It also retains
**99.2%** of the ACCESS-CM2 mean driving
signal. By contrast, the NOAA decoder fine-tune retains
**82.4%** annually and has pronounced seasonal
damping. This is evidence of climate-response forgetting during
observation-only decoder fine-tuning, not evidence that the ACCESS deployment
is inaccurate against an unavailable fine-resolution truth.

All scalar map statistics are ocean-only and cosine-latitude weighted.
Perkins skill score and SST extreme diagnostics are proposed additions and are
not silently substituted by the existing marine-heatwave proxy.


In [ ]:
from pathlib import Path
from IPython.display import Image, display

climate_figure_dir = Path(BASE_DIR if 'BASE_DIR' in globals() else '.') / 'figures/climate_change'
for filename in (
    'flow_sr_combined_ofam_climate_change_signal.png',
    'flow_sr_combined_ofam_signal_error_and_seasons.png',
    'access_cm2_signal_preservation_requested_models.png',
    'access_cm2_signal_preservation_seasonal.png',
    'ofam_combined_flow_gan_climate_signal_comparison.png',
    'access_cm2_flow_gan_training_period_comparison.png',
    'access_cm2_flow_gan_signal_ratio_comparison.png',
):
    path = climate_figure_dir / filename
    if path.is_file():
        display(Image(filename=str(path)))


In [ ]:
import json
from pathlib import Path

climate_report_path = Path(BASE_DIR if 'BASE_DIR' in globals() else '.') / 'reports/climate_change/requested_models_climate_change_evaluation.json'
climate_change_report = json.loads(climate_report_path.read_text())
climate_change_report


{
  "access_cm2": {
    "flow_sr_combined": {
      "annual_signal_preservation": {
        "mae_c": 0.025419721271782315,
        "mean_bias_c": -0.024567926044074047,
        "mean_signal_ratio": 0.9915815368550701,
        "pattern_std_ratio": 0.9878368806926359,
        "prediction_mean_c": 2.8937706853056473,
        "rmse_c": 0.029699137220704434,
        "spatial_correlation": 0.9996468583012006,
        "target_mean_c": 2.918338611349721,
        "valid_cells": 716
      },
      "files": {
        "future": "/esi/project/niwa03712/rampaln/PUBLICATIONS/2026/SSTDownscaling/runs/flow_sr_combined_hist_rcp85_continue_320k/access_cm2_converted/future_2080-01-01_2089-12-31_ab3pc_75step.nc",
        "historical": "/esi/project/niwa03712/rampaln/PUBLICATIONS/2026/SSTDownscaling/runs/flow_sr_combined_hist_rcp85_continue_320k/access_cm2_converted/historical_1980-01-01_1989-12-31_ab3pc_75step.nc"
      },
      "future_period": "2080-01-01 to 2089-12-31",
      "historical_period": "1980-